# 02 · Shared-vocabulary training labels

Read aligned cell annotations, propagate niches to image regions, and save label maps with their vocabularies. Domain conversion requires a unique domain per niche; the study annotations violate this assumption, so existing domain rasters remain separate training inputs.


## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/02_he_prediction").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "he_reference_v2")
OUT = ROOT / "outputs/he_prediction" / RUN_ID
DEVICE = os.environ.get(
    "SPAGC_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu"
)
CFG = {
    "propagation": {
        "sigma": 60,
        "truncate": 3.0,
        "min_total_weight": 0.0001,
        "min_winner_frac": 0.45,
        "min_margin": 0.0001,
        "coarse_tile_size": 128,
    }
}
VOCABULARIES = {
    "domain": [
        "Adjacent Normal area",
        "Tumor-stromal interface",
        "Low-quality",
        "Tumor Parenchyma",
        "Stroma",
    ],
    "niche": [
        "CAF enriched",
        "Chief cell enriched",
        "Epithelium",
        "Fibrotic stroma",
        "Gastric-type Tumor",
        "Gland stroma",
        "Homeostatic stroma",
        "Inflamed gastric mucosal",
        "Inflammation-infiltrated stroma",
        "Invasive Tumor nest",
        "Invasive Tumor region",
        "Low quality",
        "Neck cell enriched",
        "Normal vasculature",
        "Plasma enriched stroma",
        "Smooth muscle",
        "TAM enriched",
        "TAN & Mono enriched",
        "TLS-like structure",
        "Tumor nest",
        "Tumor vasculature",
    ],
}
LABEL_DIR = OUT / "labels/proseg1"

In [ ]:
from spagc._io import require_files
from spagc.histology.labels import convert_niche_to_domain, generate_labels

Select aligned cells, their native H&E image, and shared niche/domain vocabularies.

In [ ]:
H5AD = ROOT / "data/he_aligned_h5ad/proseg_1_aligned.h5ad"
IMAGE = ROOT / "data/he_slides/proseg1.tif"
vocabulary = VOCABULARIES
(NICHES, DOMAINS) = (vocabulary["niche"], vocabulary["domain"])
if LABEL_DIR.exists() and any(LABEL_DIR.iterdir()):
    raise FileExistsError("Label output exists; choose a fresh SPAGC_RUN_ID.")
LABEL_DIR.mkdir(parents=True, exist_ok=True)
require_files(H5AD, IMAGE)

## 2. Propagate niche annotations to image regions

Convolve class-wise cell impulses with Gaussian kernels, apply tissue/support/confidence criteria, and reduce pixel labels to regional supervision.

Propagate cell annotations to tissue-supported image regions and retain the class dictionary and raster geometry.

In [ ]:
niche_path = LABEL_DIR / "label_niche_uint16.dat"
result = generate_labels(
    str(H5AD),
    str(IMAGE),
    str(niche_path),
    column="niche",
    spatial_key="spatial_aligned",
    categories=NICHES,
    device=DEVICE,
    **CFG["propagation"],
)
(_, mapping, label_shape, he_shape) = result
print({"niche_ids": mapping, "label_shape": label_shape, "he_shape": he_shape})

## 3. Generate domain supervision and save label rasters

The original conversion requires each niche to map to one domain. It raises `ValueError` for ambiguous annotations. When the assumption holds, save the class dictionary and raster geometry, then inspect foreground support.


Create domain supervision using the aligned cell annotations and the declared domain vocabulary.

In [ ]:
domain_path = LABEL_DIR / "label_domain_uint16.dat"
convert_niche_to_domain(str(niche_path), str(domain_path), str(H5AD), NICHES, DOMAINS)
json_sidecar = {
    "dtype": "uint16",
    "shape": label_shape,
    "he_shape": he_shape,
    "region_size": 28,
    "column": "domain",
    "classes": {name: i + 1 for (i, name) in enumerate(DOMAINS)},
}
Path(str(domain_path) + ".json").write_text(json.dumps(json_sidecar, indent=2))
for path, classes in ((niche_path, NICHES), (domain_path, DOMAINS)):
    raster = np.memmap(path, dtype=np.uint16, mode="r", shape=tuple(label_shape))
    assert path.stat().st_size == int(np.prod(label_shape)) * 2
    assert raster.max() <= len(classes)
    assert np.any(raster > 0), (
        "No confident foreground labels; inspect registration and thresholds."
    )
    print(path.name, {"foreground_fraction": float(np.mean(raster > 0))})